In [ ]:
import time
import numpy as np
from pynq.overlays.base import BaseOverlay
from pynq.lib import MicroblazeLibrary
base = BaseOverlay("base.bit")

In [ ]:
liba = MicroblazeLibrary(base.PMODA, ['i2c'])

In [ ]:
device = liba.i2c_open(3, 2)

In [ ]:
# Initialize SPI
channel_0_and_1 = '00110000'
channel_0_only = '00010000'
channel_1_only = '00100000'

buf = bytearray(2)
buf[0] = int(channel_0_and_1, 2)

device.write(0x28, buf, 1)

In [ ]:
def read_and_parse(device):
    """
    Reads ADC and parses channel information
    and data into a tuple (channel, data)
    
    :param device: the MicroblazeLibrary I2C device
    
    :return: tuple of (channel, data)
    """
    buf = bytearray(2)
    device.read(0x28, buf, 2)
    channel = (buf[0] & 0x30) >> 4
    data = (((buf[0] & 0xf) << 8) | buf[1])
    return channel, data

In [ ]:
def record(device, duration) -> dict[int, np.array]:
    """
    Records data on all channels for duration seconds
    and stores data per channel in a dictionary that maps the 
    channel ID --> data.
    
    i.e. channels[1] --> [array of data recorded on channel 1]
    
    :param device: the MicroblazeLibrary I2C device
    :param duration: the amount of time to record for
    
    :return: dictionary mapping channel ID to array of data
    """
    buffer = []
    start = time.time()
    while (time.time() - start) < duration:
        buffer.append(read_and_parse(device))
    channels = {}
    for i in range(4):
        channels[i] = np.array(
            [x[1] for x in data if x[0] == i],
            dtype=np.uint16,
        )
    return channels

In [ ]:
# record for 5 seconds
data = record(device, 5)

In [ ]:
data[1]